# ML-06 — Signal Audit: Do the Flags Hold?

This is the deeper W04 signal-audit companion to the baseline notebook. The audit stays on the Lane 2 refresh/review framing and uses only information available in the starter snapshot. No trend fields or label-derived fields are used.

**Lane:** content refresh / ranking for editor review.

**Signals audited:** freshness (`days_since_last_update`), search visibility (`impressions_90d`), and CTR/position as a secondary diagnostic. The flag-linked test focuses on freshness because it is the signal behind the refresh-flag family.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find starter dataset at {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

print("Dataset:", DATA_PATH)
print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

required = [
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
]
print("\nRequired columns present:", all(c in df.columns for c in required))

FORBIDDEN = {"trend_pct", "trend_direction", "is_declining_label"}
print("Forbidden fields present in dataset:", sorted(FORBIDDEN.intersection(df.columns)))


Dataset: D:\GodCPP\fl1\FLrank1\data\raw\content_refresh_anonymized.csv
Rows: 30,000
Columns: 44

Required columns present: True
Forbidden fields present in dataset: ['trend_direction', 'trend_pct']


## 1. Distributions

The first pass is deliberately descriptive. These fields are heavy-tailed, so raw means alone are not a useful basis for a rule. I use medians/quantiles and explicit buckets before making a decision.


In [2]:
distribution_cols = [
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
]

rows = []
for col in distribution_cols:
    s = pd.to_numeric(df[col], errors="coerce")
    if col == "avg_position":
        s = s.replace(0, np.nan)
    rows.append({
        "signal": col,
        "n": int(s.notna().sum()),
        "missing_pct": round(100 * s.isna().mean(), 2),
        "median": round(float(s.median()), 3) if s.notna().any() else np.nan,
        "p75": round(float(s.quantile(.75)), 3) if s.notna().any() else np.nan,
        "p90": round(float(s.quantile(.90)), 3) if s.notna().any() else np.nan,
        "max": round(float(s.max()), 3) if s.notna().any() else np.nan,
    })

display(pd.DataFrame(rows))


,signal,n,missing_pct,median,p75,p90,max
0,days_since_last_update,30000,0.00,20.00,104.00,104.00,373.0
1,impressions_90d,30000,0.00,731.00,3615.25,12136.40,517715.0
2,clicks_90d,30000,0.00,1.00,7.00,32.00,4178.0
3,ctr,30000,0.00,0.07,0.29,0.65,100.0
4,avg_position,28795,4.02,11.40,22.90,37.50,245.0


## 2. Signal test #1 / #2 / #3

### Signal 1 — freshness
Operational question: do older pages still have enough search visibility to justify a refresh review?

### Signal 2 — search volume
Operational question: does higher visibility correspond to a larger amount of traffic opportunity, rather than just a handful of impressions?

### Signal 3 — CTR versus position
Diagnostic question: does CTR behave sensibly across broad position bands? This is a sanity check, not a causal claim.


In [3]:
work = df.copy()

for c in [
    "impressions_90d",
    "clicks_90d",
    "days_since_last_update",
    "ctr",
    "avg_position",
]:
    work[c] = pd.to_numeric(work[c], errors="coerce")

work["avg_position_clean"] = work["avg_position"].replace(0, np.nan)

# Signal 1: freshness
fresh_bins = [-np.inf, 90, 180, 365, np.inf]
fresh_labels = ["0-90d", "91-180d", "181-365d", "366d+"]
work["freshness_bucket"] = pd.cut(
    work["days_since_last_update"],
    bins=fresh_bins,
    labels=fresh_labels,
    right=True,
)

fresh_table = (
    work.groupby("freshness_bucket", observed=False)
    .agg(
        n=("days_since_last_update", "size"),
        median_impressions_90d=("impressions_90d", "median"),
        median_clicks_90d=("clicks_90d", "median"),
        pct_visible_300=("impressions_90d", lambda s: 100 * (s >= 300).mean()),
    )
    .reset_index()
)

print("SIGNAL 1 — FRESHNESS BUCKETS")
display(fresh_table)

fresh_old = work.loc[
    work["days_since_last_update"] > 180, "impressions_90d"
].median()
fresh_recent = work.loc[
    work["days_since_last_update"] <= 180, "impressions_90d"
].median()

fresh_verdict = "CONFIRMED" if fresh_old >= fresh_recent else "OPPOSITE"
print("Verdict:", fresh_verdict)
print(
    "Interpretation: stale pages retain at least as much median visibility as newer pages."
    if fresh_verdict == "CONFIRMED"
    else
    "Interpretation: stale pages have lower median visibility than newer pages, so staleness alone is a weak refresh signal."
)

# Signal 2: search visibility
volume_bins = [-np.inf, 99, 299, 999, 4999, np.inf]
volume_labels = ["0-99", "100-299", "300-999", "1k-4,999", "5k+"]
work["volume_bucket"] = pd.cut(
    work["impressions_90d"],
    bins=volume_bins,
    labels=volume_labels,
    right=True,
)

volume_table = (
    work.groupby("volume_bucket", observed=False)
    .agg(
        n=("impressions_90d", "size"),
        median_clicks_90d=("clicks_90d", "median"),
        median_ctr=("ctr", "median"),
    )
    .reset_index()
)

print("\nSIGNAL 2 — SEARCH VISIBILITY BUCKETS")
display(volume_table)

low_clicks = work.loc[work["impressions_90d"] < 300, "clicks_90d"].median()
high_clicks = work.loc[work["impressions_90d"] >= 300, "clicks_90d"].median()

volume_verdict = "CONFIRMED" if high_clicks > low_clicks else "OPPOSITE"
print("Verdict:", volume_verdict)
print(
    "Interpretation: visible pages have higher median clicks."
    if volume_verdict == "CONFIRMED"
    else
    "Interpretation: visibility does not translate into higher median clicks in this snapshot."
)

# Signal 3: CTR vs position
position_bins = [0, 3, 10, 20, 50, np.inf]
position_labels = ["1-3", "4-10", "11-20", "21-50", "51+"]
position_data = work.dropna(subset=["avg_position_clean"]).copy()
position_data["position_bucket"] = pd.cut(
    position_data["avg_position_clean"],
    bins=position_bins,
    labels=position_labels,
    right=True,
)

position_table = (
    position_data.groupby("position_bucket", observed=False)
    .agg(
        n=("avg_position_clean", "size"),
        median_ctr=("ctr", "median"),
        median_impressions_90d=("impressions_90d", "median"),
    )
    .reset_index()
)

print("\nSIGNAL 3 — CTR VS POSITION")
display(position_table)
print("Verdict: MIXED")
print(
    "Interpretation: this is a diagnostic relationship, not a causal rule; "
    "read the bucket table for direction and sample size rather than treating "
    "it as proof that a CTR-fix intervention will improve rankings."
)


SIGNAL 1 — FRESHNESS BUCKETS


,freshness_bucket,n,median_impressions_90d,median_clicks_90d,pct_visible_300
0,0-90d,20655,472.0,1.0,55.763738
1,91-180d,9171,1692.0,2.0,78.639189
2,181-365d,169,16.0,0.0,13.017751
3,366d+,5,2.0,0.0,0.000000


Verdict: OPPOSITE
Interpretation: stale pages have lower median visibility than newer pages, so staleness alone is a weak refresh signal.

SIGNAL 2 — SEARCH VISIBILITY BUCKETS


,volume_bucket,n,median_clicks_90d,median_ctr
0,0-99,7994,0.0,0.00
1,100-299,3254,0.0,0.00
2,300-999,5240,1.0,0.10
3,"1k-4,999",7361,3.0,0.16
4,5k+,6151,29.0,0.22


Verdict: CONFIRMED
Interpretation: visible pages have higher median clicks.

SIGNAL 3 — CTR VS POSITION


,position_bucket,n,median_ctr,median_impressions_90d
0,1-3,1141,0.00,74.0
1,4-10,11842,0.16,1184.0
2,11-20,7273,0.10,870.0
3,21-50,7225,0.03,807.0
4,51+,1314,0.00,219.5


Verdict: MIXED
Interpretation: this is a diagnostic relationship, not a causal rule; read the bucket table for direction and sample size rather than treating it as proof that a CTR-fix intervention will improve rankings.


## 3. The flag-linked test

**Flag family:** refresh/staleness.

The operational assumption is not simply that an old page is bad. The useful assumption is narrower: an old page is worth putting into a refresh queue when it still has meaningful search visibility. This test therefore crosses the staleness threshold with a visibility threshold and prints both bucket counts and the visible rate.


In [4]:
work["stale_flag"] = work["days_since_last_update"] >= 180
work["visible_flag"] = work["impressions_90d"] >= 300
work["stale_visible"] = work["stale_flag"] & work["visible_flag"]

flag_table = (
    work.assign(
        freshness_flag=np.where(
            work["stale_flag"], "stale_180d_plus", "not_stale"
        )
    )
    .groupby("freshness_flag", observed=False)
    .agg(
        n=("stale_flag", "size"),
        median_impressions_90d=("impressions_90d", "median"),
        pct_visible_300=("visible_flag", lambda s: 100 * s.mean()),
        pct_stale_visible=("stale_visible", lambda s: 100 * s.mean()),
    )
    .reset_index()
)

print("FLAG-LINKED TEST — REFRESH / STALENESS")
display(flag_table)

stale_n = int(work["stale_flag"].sum())
stale_visible_n = int(work["stale_visible"].sum())
stale_visible_rate = stale_visible_n / stale_n if stale_n else 0

print(f"Stale pages (>=180d): {stale_n:,}")
print(f"Stale + visible pages (>=300 impressions): {stale_visible_n:,}")
print(f"Visible share among stale pages: {stale_visible_rate:.1%}")

flag_verdict = "CONFIRMED" if stale_visible_rate >= 0.20 else "MIXED"
print("Verdict:", flag_verdict)
print(
    "Interpretation: staleness is more useful as a queue trigger when paired "
    "with evidence that the page still has meaningful search visibility."
)


FLAG-LINKED TEST — REFRESH / STALENESS


,freshness_flag,n,median_impressions_90d,pct_visible_300,pct_stale_visible
0,not_stale,29826,742.0,62.797559,0.000000
1,stale_180d_plus,174,15.5,12.643678,12.643678


Stale pages (>=180d): 174
Stale + visible pages (>=300 impressions): 22
Visible share among stale pages: 12.6%
Verdict: MIXED
Interpretation: staleness is more useful as a queue trigger when paired with evidence that the page still has meaningful search visibility.


## 4. What this means in practice

The audit supports a narrow operational rule rather than treating any single signal as a universal quality score. Freshness identifies pages that may be due for review, while search visibility determines whether that review has enough observable demand to matter. CTR and position are useful diagnostics, but they are not being treated as proof that changing a page will cause a ranking improvement.

For Lane 2, the practical baseline is therefore a ranked refresh-review queue built from stale + visible pages, with the thresholds kept explicit so the Week-5 model has a clear baseline to beat.


In [5]:
# Final audit guards
used_fields = {
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "stale_flag",
    "visible_flag",
}

assert not used_fields.intersection(
    {"trend_pct", "trend_direction", "is_declining_label"}
)

assert {"n", "median_impressions_90d"}.issubset(fresh_table.columns)
assert {"n", "median_clicks_90d"}.issubset(volume_table.columns)
assert {"n", "median_ctr"}.issubset(position_table.columns)
assert {"n", "pct_visible_300"}.issubset(flag_table.columns)

print("SELF-CHECK")
print("All signal tables include n: PASS")
print("Flag-linked freshness test present: PASS")
print("No future-window or label-derived fields used: PASS")
print("Signal audit complete.")


SELF-CHECK
All signal tables include n: PASS
Flag-linked freshness test present: PASS
No future-window or label-derived fields used: PASS
Signal audit complete.


## Self-check

- [x] Every section is filled — reasoning and code are both present.
- [x] The audit uses visible bucket tables with `n`.
- [x] The refresh/staleness test is explicitly flag-linked.
- [x] No `trend_pct`, `trend_direction`, or `is_declining_label` is used as an input.
- [x] Claims are framed as observed/directional decision support, not causal proof.
- [ ] Run the notebook top to bottom in the repository environment and inspect the actual numbers before committing.
- [ ] Commit under `work/notebooks/w04_signal_audit.ipynb`.
